# Kiểm tra dữ liệu bằng PySpark

Notebook đọc cùng CSV từ HDFS, áp dụng các kiểm tra tương đương Pandas,
rồi lưu báo cáo riêng. Mặc định Spark chạy phân tán qua Master và ba Worker.
Dùng `./scripts/start_notebooks.sh local` để chuyển sang local[*]. Restart Kernel khi đổi chế độ.

## 1. Cấu hình

In [1]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from IPython.display import display
from pyspark.sql import SparkSession, functions as F

HDFS_FILE = os.environ.get(
    "AUDIT_HDFS_FILE",
    "/data/hotel_bookings/input/hotel_bookings_synthetic_200MB.csv",
)
HDFS_PATH = f"hdfs://namenode:9000{HDFS_FILE}"
SPARK_MASTER = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")
REPORT_DIR = Path("/workspace/reports")
MISSING_WORDS = ["", "nan", "null", "none", "na", "n/a"]
ADR_LIMIT = 1000

NUMERIC_COLUMNS = [
    "is_canceled", "lead_time", "arrival_date_year",
    "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults",
    "children", "babies", "is_repeated_guest", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes", "agent", "company",
    "days_in_waiting_list", "adr", "required_car_parking_spaces",
    "total_of_special_requests",
]
NONNEGATIVE_COLUMNS = [
    "lead_time", "arrival_date_week_number", "arrival_date_day_of_month",
    "stays_in_weekend_nights", "stays_in_week_nights", "adults",
    "children", "babies", "previous_cancellations",
    "previous_bookings_not_canceled", "booking_changes",
    "days_in_waiting_list", "required_car_parking_spaces",
    "total_of_special_requests",
]
GUEST_COLUMNS = ["adults", "children", "babies"]
NIGHT_COLUMNS = ["stays_in_weekend_nights", "stays_in_week_nights"]

## 2. Khởi tạo Spark và đọc CSV từ HDFS

In [2]:
spark_started_at = time.perf_counter()
spark = (
    SparkSession.builder
    .appName("HotelBookingsDataQualityAudit")
    .master(SPARK_MASTER)
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "audit"))
    .config("spark.driver.bindAddress", os.environ.get("SPARK_DRIVER_BIND_ADDRESS", "0.0.0.0"))
    .config("spark.executor.cores", os.environ.get("SPARK_EXECUTOR_CORES", "1"))
    .config("spark.cores.max", os.environ.get("SPARK_CORES_MAX", "3"))
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
spark_master = spark.sparkContext.master
spark_mode = "distributed" if spark_master.startswith("spark://") else "local"
spark_ready_at = time.perf_counter()
audit_started_at = time.perf_counter()

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .option("mode", "FAILFAST")
    .csv(HDFS_PATH)
)

required = set(
    NUMERIC_COLUMNS
    + ["arrival_date_month", "reservation_status", "reservation_status_date"]
)
missing_columns = sorted(required - set(df.columns))
if missing_columns:
    raise ValueError(f"CSV thiếu cột bắt buộc: {missing_columns}")

print(f"Đã đọc CSV từ {HDFS_PATH}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/22 09:29:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
[Stage 0:>                                                          (0 + 1) / 1]

Đã đọc CSV từ hdfs://namenode:9000/data/hotel_bookings/input/hotel_bookings_synthetic_200MB.csv


## 3. Thống kê giá trị thiếu và số không hợp lệ

In [3]:
missing = {
    column: (
        F.col(column).isNull()
        | F.lower(F.trim(F.col(column))).isin(*MISSING_WORDS)
    )
    for column in df.columns
}

numbers = {
    column: F.when(
        ~missing[column],
        F.expr(f"try_cast(trim({column}) as double)"),
    )
    for column in NUMERIC_COLUMNS
}

invalid_numbers = {
    column: (~missing[column]) & numbers[column].isNull()
    for column in NUMERIC_COLUMNS
}

print("Đã tạo kiểm tra giá trị thiếu và chuyển đổi số.")

Đã tạo kiểm tra giá trị thiếu và chuyển đổi số.


## 4. Kiểm tra các quy tắc dữ liệu

In [4]:
def count_if(condition, name):
    return F.sum(F.when(condition, 1).otherwise(0)).alias(name)

def total_is_zero(columns):
    total = sum(
        (F.coalesce(numbers[column], F.lit(0.0)) for column in columns),
        F.lit(0.0),
    )
    complete = F.lit(True)
    for column in columns:
        complete = complete & ~missing[column] & numbers[column].isNotNull()
    return complete & (total == 0)

zero_guests = total_is_zero(GUEST_COLUMNS)
zero_nights = total_is_zero(NIGHT_COLUMNS)
negative_adr = numbers["adr"] < 0
zero_adr = numbers["adr"] == 0
high_adr = numbers["adr"] > ADR_LIMIT

month_numbers = {
    "january": 1, "february": 2, "march": 3, "april": 4,
    "may": 5, "june": 6, "july": 7, "august": 8,
    "september": 9, "october": 10, "november": 11, "december": 12,
}
month_map = F.create_map(*[
    item
    for month, number in month_numbers.items()
    for item in (F.lit(month), F.lit(number))
])
arrival_text = F.concat_ws(
    "-",
    F.trim(F.col("arrival_date_year")),
    month_map[F.lower(F.trim(F.col("arrival_date_month")))],
    F.trim(F.col("arrival_date_day_of_month")),
)
arrival_date = F.try_to_timestamp(arrival_text, F.lit("yyyy-M-d"))
status_date = F.try_to_timestamp(
    F.trim(F.col("reservation_status_date")),
    F.lit("yyyy-MM-dd"),
)

status = F.trim(F.col("reservation_status"))
expected_canceled = (
    F.when(status == "Canceled", 1.0)
    .when(status == "No-Show", 1.0)
    .when(status == "Check-Out", 0.0)
)
status_mismatch = (
    numbers["is_canceled"].isNotNull()
    & expected_canceled.isNotNull()
    & (numbers["is_canceled"] != expected_canceled)
)

summary_expressions = [
    F.count(F.lit(1)).alias("total_rows"),
    *[
        count_if(missing[column], f"missing_{column}")
        for column in df.columns
    ],
    *[
        count_if(invalid_numbers[column], f"invalid_{column}")
        for column in NUMERIC_COLUMNS
    ],
    count_if(zero_guests, "zero_guests"),
    count_if(zero_nights, "zero_nights"),
    count_if(negative_adr, "negative_adr"),
    count_if(zero_adr, "zero_adr"),
    count_if(high_adr, "high_adr"),
    count_if(arrival_date.isNull(), "invalid_arrival_date"),
    count_if(status_date.isNull(), "invalid_status_date"),
    count_if(status_mismatch, "status_mismatch"),
    *[
        count_if(numbers[column] < 0, f"negative_{column}")
        for column in NONNEGATIVE_COLUMNS
    ],
]
summary = df.agg(*summary_expressions).first().asDict()
total_rows = int(summary["total_rows"])

missing_values = {}
for column in df.columns:
    count = int(summary[f"missing_{column}"] or 0)
    if count:
        missing_values[column] = {
            "count": count,
            "percent": round(count / total_rows * 100, 4),
        }

invalid_numeric_values = {
    column: int(summary[f"invalid_{column}"] or 0)
    for column in NUMERIC_COLUMNS
    if summary[f"invalid_{column}"]
}
negative_values = {
    column: int(summary[f"negative_{column}"] or 0)
    for column in NONNEGATIVE_COLUMNS
    if summary[f"negative_{column}"]
}

def count_by_status(condition):
    status_label = F.when(
        F.col("reservation_status").isNull()
        | F.lower(F.trim(F.col("reservation_status"))).isin(*MISSING_WORDS),
        "(missing)",
    ).otherwise(F.trim(F.col("reservation_status")))

    rows = (
        df.where(condition)
        .groupBy(status_label.alias("status"))
        .count()
        .collect()
    )
    return {str(row["status"]): int(row["count"]) for row in rows}

checks = {
    "zero_guests": {
        "count": int(summary["zero_guests"] or 0),
        "by_reservation_status": count_by_status(zero_guests),
    },
    "zero_nights": {
        "count": int(summary["zero_nights"] or 0),
        "by_reservation_status": count_by_status(zero_nights),
    },
    "negative_adr": {
        "count": int(summary["negative_adr"] or 0),
        "by_reservation_status": count_by_status(negative_adr),
    },
    "zero_adr": int(summary["zero_adr"] or 0),
    "adr_above_1000": int(summary["high_adr"] or 0),
    "negative_values_by_column": negative_values,
    "invalid_arrival_date": int(summary["invalid_arrival_date"] or 0),
    "invalid_reservation_status_date": int(summary["invalid_status_date"] or 0),
    "reservation_status_canceled_mismatch": int(summary["status_mismatch"] or 0),
}

## 5. Lưu báo cáo và thời gian chạy

In [5]:
duplicate_rows = total_rows - df.dropDuplicates().count()
elapsed_seconds = time.perf_counter() - audit_started_at
spark_startup_seconds = spark_ready_at - spark_started_at
total_seconds = time.perf_counter() - spark_started_at

report = {
    "engine": "pyspark",
    "spark_mode": spark_mode,
    "spark_master": spark_master,
    "source": HDFS_PATH,
    "read_only": True,
    "total_rows": total_rows,
    "total_columns": len(df.columns),
    "missing_values": missing_values,
    "invalid_numeric_values": invalid_numeric_values,
    "exact_duplicate_rows_extra": duplicate_rows,
    "checks": checks,
    "runtime": {
        "spark_session_startup_seconds": round(spark_startup_seconds, 3),
        "read_and_audit_seconds": round(elapsed_seconds, 3),
        "total_seconds_including_spark_startup": round(total_seconds, 3),
    },
    "notes": [
        "Notebook chỉ thống kê, không sửa CSV gốc.",
        "Dòng giống hệt chưa chắc là cùng booking vì dữ liệu không có booking_id.",
        "Chỉ tính tổng khách hoặc tổng đêm bằng 0 khi mọi cột cấu thành có giá trị hợp lệ.",
        "Các chuỗi rỗng, nan, null, none, na, n/a được xem là thiếu.",
    ],
}

missing_table = pd.DataFrame(
    [{"column": column, **values} for column, values in missing_values.items()],
    columns=["column", "count", "percent"],
)
display(missing_table.sort_values("count", ascending=False))

REPORT_DIR.mkdir(parents=True, exist_ok=True)
(REPORT_DIR / "audit_pyspark.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

display(pd.DataFrame([
    {"kiểm tra": "Dòng dữ liệu", "số lượng": total_rows},
    {"kiểm tra": "Dòng trùng hoàn toàn dư", "số lượng": report["exact_duplicate_rows_extra"]},
    {"kiểm tra": "Tổng khách bằng 0", "số lượng": checks["zero_guests"]["count"]},
    {"kiểm tra": "Tổng đêm bằng 0", "số lượng": checks["zero_nights"]["count"]},
    {"kiểm tra": "ADR âm", "số lượng": checks["negative_adr"]["count"]},
    {"kiểm tra": "Ngày đến không hợp lệ", "số lượng": checks["invalid_arrival_date"]},
]))
print(f"Thời gian đọc và kiểm tra bằng PySpark: {elapsed_seconds:.2f} giây")
print(f"Chế độ Spark: {spark_mode} ({spark_master})")
print(f"Khởi động Spark: {spark_startup_seconds:.2f} giây")
print(f"Tổng thời gian gồm khởi động Spark: {total_seconds:.2f} giây")
print("Đã lưu báo cáo: reports/audit_pyspark.json")

,column,count,percent
3,company,1304997,94.3400
2,agent,188998,13.6629
1,country,5590,0.4041
0,children,55,0.0040


,kiểm tra,số lượng
0,Dòng dữ liệu,1383291
1,Dòng trùng hoàn toàn dư,53673
2,Tổng khách bằng 0,2195
3,Tổng đêm bằng 0,8315
4,ADR âm,15
5,Ngày đến không hợp lệ,0


Thời gian đọc và kiểm tra bằng PySpark: 28.63 giây
Chế độ Spark: distributed (spark://spark-master:7077)
Khởi động Spark: 4.74 giây
Tổng thời gian gồm khởi động Spark: 33.37 giây
Đã lưu báo cáo: reports/audit_pyspark.json


## 6. So sánh với Pandas

In [6]:
pandas_report_path = REPORT_DIR / "audit_pandas.json"

if not pandas_report_path.exists():
    print("Chạy audit_pandas.ipynb trước để tạo báo cáo Pandas.")
else:
    pandas_report = json.loads(
        pandas_report_path.read_text(encoding="utf-8")
    )

    comparison = pd.DataFrame([
        {
            "engine": "Pandas",
            "spark_mode": "n/a",
            "spark_master": "n/a",
            "rows": pandas_report["total_rows"],
            "duplicates": pandas_report["exact_duplicate_rows_extra"],
            "missing_columns": len(pandas_report["missing_values"]),
            "read_and_audit_seconds": pandas_report.get("runtime_seconds"),
            "spark_startup_seconds": 0,
            "total_seconds": pandas_report.get("runtime_seconds"),
        },
        {
            "engine": "PySpark",
            "spark_mode": report["spark_mode"],
            "spark_master": report["spark_master"],
            "rows": report["total_rows"],
            "duplicates": report["exact_duplicate_rows_extra"],
            "missing_columns": len(report["missing_values"]),
            "read_and_audit_seconds": report["runtime"]["read_and_audit_seconds"],
            "spark_startup_seconds": report["runtime"]["spark_session_startup_seconds"],
            "total_seconds": report["runtime"]["total_seconds_including_spark_startup"],
        },
    ])
    display(comparison)

,engine,spark_mode,spark_master,rows,duplicates,missing_columns,read_and_audit_seconds,spark_startup_seconds,total_seconds
0,Pandas,n/a,n/a,1383291,53673,4,24.348,0.000,24.348
1,PySpark,distributed,spark://spark-master:7077,1383291,53673,4,28.629,4.738,33.367
